# Zepto Data Pipeline Assignment — books.toscrape.com
**Module:** `/data_pipeline`
**What this notebook does:** scrape → clean → convert (GBP→INR fixed rate) → load into normalized SQLite → query with SQL → cross-check with pandas.

> Note on this run: books.toscrape.com could not be reached from the machine this notebook was executed on (blocked outbound network), so the scraper automatically falls back to a local HTML mirror (`fixtures.py`) that uses the **exact same markup/CSS classes** as the real site. The scraping/parsing code itself is identical either way — on a machine with normal internet access, `requests.get()` succeeds and the fallback branch is simply never used. This is printed as a `[warning]` below so it's obvious when it happens.


In [1]:
!pip3 install pandas
!pip3 insatll beautifulsoup


[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: unknown command "insatll" - maybe you meant "install"



In [2]:
import sys, os, re, sqlite3
import requests
from bs4 import BeautifulSoup
import pandas as pd

from fixtures import get_all_fixture_pages, CATEGORIES   # local mirror, explained above

GBP_TO_INR = 105.50   # <-- fixed project baseline rate (assignment requirement), not a live lookup
BASE_URL = "https://books.toscrape.com"
RATING_WORD_TO_NUM = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

FIXTURE_PAGES = get_all_fixture_pages()
USED_FALLBACK = False
print("categories I'm going to scrape:", list(CATEGORIES.keys()))

categories I'm going to scrape: ['Mystery', 'Fiction', 'Fantasy', 'Science']


## Step 1: fetch_page() — try the real internet first, fall back to local mirror only if it fails

In [3]:
def fetch_page(url):
    """Try to actually download the page with requests.
    Falls back to the local fixture HTML only if the real request fails
    (e.g. no internet). On a normal machine this fallback never triggers.
    """
    global USED_FALLBACK
    try:
        resp = requests.get(url, timeout=6)
        if resp.status_code == 200:
            return resp.text
        else:
            raise Exception(f"bad status code {resp.status_code}")
    except Exception as e:
        if not USED_FALLBACK:
            print(f"[warning] could not reach the internet ({e}).")
            print("[warning] using local fixture HTML instead so the pipeline still runs.")
            USED_FALLBACK = True
        if url in FIXTURE_PAGES:
            return FIXTURE_PAGES[url]
        raise

## Step 2: scrape_category() — parses one category, follows pagination via the 'next' link

In [4]:
def scrape_category(category_name, start_url):
    rows = []
    url = start_url
    page_num = 1
    while url:
        html = fetch_page(url)
        soup = BeautifulSoup(html, "html.parser")
        books = soup.select("article.product_pod")
        print(f"  page {page_num} of {category_name}: found {len(books)} books")

        for b in books:
            title = b.h3.a["title"]
            price_text = b.select_one("p.price_color").get_text(strip=True)
            star_tag = b.select_one("p.star-rating")
            star_word = star_tag["class"][1] if star_tag and len(star_tag["class"]) > 1 else None
            availability_text = b.select_one("p.instock.availability").get_text(strip=True)

            rows.append({
                "title": title,
                "price": price_text,
                "star_rating": star_word,
                "availability": availability_text,
                "category": category_name,
            })

        # real site pagination: look for a "next" page link
        next_link = soup.select_one("li.next a")
        if next_link:
            url = start_url.rsplit("/", 1)[0] + "/" + next_link["href"]
            page_num += 1
        else:
            url = None
    return rows


def scrape_all_categories(categories_dict):
    all_rows = []
    for cat_name, url in categories_dict.items():
        print(f"scraping category: {cat_name}")
        all_rows.extend(scrape_category(cat_name, url))
    return all_rows

## Step 3: run the scrape across all 4 categories (need ≥60 books across ≥3 categories — got 72 across 4)

In [5]:
raw_rows = scrape_all_categories(CATEGORIES)
raw_df = pd.DataFrame(raw_rows)
print(f"\nTotal raw rows scraped: {len(raw_df)}")
print(f"Categories scraped: {raw_df['category'].nunique()} -> {list(raw_df['category'].unique())}")
raw_df.head()

scraping category: Mystery
  page 1 of Mystery: found 20 books
  page 2 of Mystery: found 12 books
scraping category: Fiction
  page 1 of Fiction: found 20 books
  page 2 of Fiction: found 20 books
  page 3 of Fiction: found 20 books
  page 4 of Fiction: found 5 books
scraping category: Fantasy
  page 1 of Fantasy: found 20 books
  page 2 of Fantasy: found 20 books
  page 3 of Fantasy: found 8 books
scraping category: Science
  page 1 of Science: found 14 books

Total raw rows scraped: 159
Categories scraped: 4 -> ['Mystery', 'Fiction', 'Fantasy', 'Science']


,title,price,star_rating,availability,category
0,Sharp Objects,Â£47.82,Four,In stock,Mystery
1,"In a Dark, Dark Wood",Â£19.63,One,In stock,Mystery
2,The Past Never Ends,Â£56.50,Four,In stock,Mystery
3,A Murder in Time,Â£16.64,One,In stock,Mystery
4,The Murder of Roger Ackroyd (Hercule Poirot #4),Â£44.10,Four,In stock,Mystery


## Step 4: clean the raw fields
* **price_gbp** — strip currency symbols/junk, keep the number, convert to float.
* **rating** — map word (`One..Five`) to int `1..5`.
* **in_stock** — text → boolean.
* **error handling** (as required by the assignment, with justification):
  * price: a garbled/missing price is a *display* problem, not a reason to lose a real book row → **median-impute**.
  * rating: an unrecognised rating word means I genuinely don't know the rating and can't safely guess one out of thin air → **drop the row**.


In [6]:
df = raw_df.copy()

def parse_price(price_text):
    match = re.search(r"[\d]+\.?[\d]*", price_text.replace(",", ""))
    if match:
        try:
            return float(match.group())
        except ValueError:
            return None
    return None

df["price_gbp"] = df["price"].apply(parse_price)
n_missing_price = df["price_gbp"].isna().sum()
print(f"rows with unparseable price: {n_missing_price}")

if n_missing_price > 0:
    median_price = df["price_gbp"].median()
    df["price_gbp"] = df["price_gbp"].fillna(median_price)
    print(f"imputed {n_missing_price} missing price(s) with median = {median_price:.2f}")

df["rating"] = df["star_rating"].map(RATING_WORD_TO_NUM)
n_bad_rating = df["rating"].isna().sum()
print(f"rows with unrecognised rating word: {n_bad_rating} -> dropping them")
df = df.dropna(subset=["rating"]).copy()
df["rating"] = df["rating"].astype(int)

df["in_stock"] = df["availability"].str.lower().str.contains("in stock")
df["price_inr"] = (df["price_gbp"] * GBP_TO_INR).round(2)   # fixed-rate conversion, no API call

clean_df = df[["title", "category", "price_gbp", "price_inr", "rating", "in_stock"]].reset_index(drop=True)
print(f"\nfinal clean row count: {len(clean_df)}")
print(clean_df.dtypes)
clean_df.head()

rows with unparseable price: 0
rows with unrecognised rating word: 0 -> dropping them

final clean row count: 159
title         object
category      object
price_gbp    float64
price_inr    float64
rating         int64
in_stock        bool
dtype: object


,title,category,price_gbp,price_inr,rating,in_stock
0,Sharp Objects,Mystery,47.82,5045.01,4,True
1,"In a Dark, Dark Wood",Mystery,19.63,2070.96,1,True
2,The Past Never Ends,Mystery,56.50,5960.75,4,True
3,A Murder in Time,Mystery,16.64,1755.52,1,True
4,The Murder of Roger Ackroyd (Hercule Poirot #4),Mystery,44.10,4652.55,4,True


## Step 5: load into a normalized SQLite database
Two tables, `categories` and `books`, linked by `category_id` (PK in `categories`, FK in `books`).

In [7]:
DB_PATH = "books.db"
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)   # rebuild fresh every time this notebook runs

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

cur.execute("""
CREATE TABLE categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT UNIQUE NOT NULL
)
""")

cur.execute("""
CREATE TABLE books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    price_gbp REAL NOT NULL,
    price_inr REAL NOT NULL,
    rating INTEGER NOT NULL,
    in_stock INTEGER NOT NULL,
    category_id INTEGER NOT NULL,
    FOREIGN KEY (category_id) REFERENCES categories(category_id)
)
""")

category_ids = {}
for cat_name in sorted(clean_df["category"].unique()):
    cur.execute("INSERT INTO categories (category_name) VALUES (?)", (cat_name,))
    category_ids[cat_name] = cur.lastrowid

book_rows = []
for _, r in clean_df.iterrows():
    book_rows.append((
        r["title"], float(r["price_gbp"]), float(r["price_inr"]),
        int(r["rating"]), int(bool(r["in_stock"])), category_ids[r["category"]],
    ))

cur.executemany("""
    INSERT INTO books (title, price_gbp, price_inr, rating, in_stock, category_id)
    VALUES (?, ?, ?, ?, ?, ?)
""", book_rows)
conn.commit()
print(f"inserted {len(category_ids)} categories and {len(book_rows)} books into {DB_PATH}")

inserted 4 categories and 159 books into books.db


## Step 6: SQL queries
5 queries covering `SELECT/WHERE`, `ORDER BY` + `LIMIT`, `DISTINCT`, `BETWEEN` + `IN`, and a `JOIN`.

In [8]:
queries = {}

queries["Q1_instock_high_rated"] = """
SELECT title, rating, price_inr
FROM books
WHERE in_stock = 1 AND rating >= 4
"""

queries["Q2_top10_expensive"] = """
SELECT title, price_inr
FROM books
ORDER BY price_inr DESC
LIMIT 10
"""

queries["Q3_distinct_ratings"] = """
SELECT DISTINCT rating
FROM books
ORDER BY rating
"""

queries["Q4_between_and_in"] = """
SELECT b.title, b.price_inr, c.category_name
FROM books b
JOIN categories c ON b.category_id = c.category_id
WHERE b.price_inr BETWEEN 2000 AND 4000
  AND c.category_name IN ('Mystery', 'Fantasy')
"""

queries["Q5_top_rated_per_category_JOIN"] = """
SELECT c.category_name, b.title, b.rating, b.price_inr
FROM books b
JOIN categories c ON b.category_id = c.category_id
WHERE b.rating = (
    SELECT MAX(b2.rating) FROM books b2 WHERE b2.category_id = b.category_id
)
ORDER BY c.category_name, b.price_inr DESC
"""

query_results = {}
for name, sql in queries.items():
    print(f"--- {name} ---")
    print(sql.strip())
    result = cur.execute(sql).fetchall()
    cols = [d[0] for d in cur.description]
    result_df = pd.DataFrame(result, columns=cols)
    query_results[name] = result_df
    print(f"-> {len(result_df)} rows")
    print(result_df.head(10).to_string(index=False))
    print()

--- Q1_instock_high_rated ---
SELECT title, rating, price_inr
FROM books
WHERE in_stock = 1 AND rating >= 4
-> 67 rows
                                                                   title  rating  price_inr
                                                           Sharp Objects       4    5045.01
                                                     The Past Never Ends       4    5960.75
                         The Murder of Roger Ackroyd (Hercule Poirot #4)       4    4652.55
                                  A Time of Torment (Charlie Parker #14)       5    5100.92
                   Murder at the 42nd Street Library (Raymond Ambler #1)       4    5734.98
       What Happened on Beale Street (Secrets of the South Mysteries #2)       5    2676.54
The Bachelor Girl's Guide to Murder (Herringford and Watts Mysteries #1)       5    5517.65
                        Delivering the Truth (Quaker Midwife Mystery #1)       4    2203.90
                     The Mysterious Affair at Styles 

## Step 7: read results back with `pd.read_sql`, and reproduce the JOIN with `pd.merge` (no SQL) to prove they match

In [9]:
df_q1_pd = pd.read_sql(queries["Q1_instock_high_rated"], conn)
df_q5_pd = pd.read_sql(queries["Q5_top_rated_per_category_JOIN"], conn)
print("pd.read_sql(Q1) shape:", df_q1_pd.shape)
print("pd.read_sql(Q5 - the JOIN query) shape:", df_q5_pd.shape)
df_q1_pd.head()

pd.read_sql(Q1) shape: (67, 3)
pd.read_sql(Q5 - the JOIN query) shape: (34, 4)


,title,rating,price_inr
0,Sharp Objects,4,5045.01
1,The Past Never Ends,4,5960.75
2,The Murder of Roger Ackroyd (Hercule Poirot #4),4,4652.55
3,A Time of Torment (Charlie Parker #14),5,5100.92
4,Murder at the 42nd Street Library (Raymond Amb...,4,5734.98


In [10]:
categories_df_mem = pd.read_sql("SELECT * FROM categories", conn)
books_df_mem = pd.read_sql("SELECT * FROM books", conn)

merged = books_df_mem.merge(categories_df_mem, on="category_id", how="inner")

max_rating_per_cat = merged.groupby("category_id")["rating"].transform("max")
pandas_join_result = merged[merged["rating"] == max_rating_per_cat][
    ["category_name", "title", "rating", "price_inr"]
].sort_values(["category_name", "price_inr"], ascending=[True, False]).reset_index(drop=True)

sql_join_result = df_q5_pd.sort_values(["category_name", "price_inr"], ascending=[True, False]).reset_index(drop=True)

print("SQL JOIN result (via pd.read_sql):")
print(sql_join_result.to_string(index=False))
print()
print("pandas merge result (no SQL at all):")
print(pandas_join_result.to_string(index=False))

are_equal = sql_join_result.reset_index(drop=True).equals(pandas_join_result.reset_index(drop=True))
print(f"\nSQL result and pandas-merge result match exactly: {are_equal}")

SQL JOIN result (via pd.read_sql):
category_name                                                                                                                               title  rating  price_inr
      Fantasy                                                                                        The False Prince (The Ascendance Trilogy #1)       5    5908.00
      Fantasy                                                                                               Paper and Fire (The Great Library #2)       5    5216.98
      Fantasy                                                                            Harry Potter and the Half-Blood Prince (Harry Potter #6)       5    5143.12
      Fantasy                                                                                            The Beast (Black Dagger Brotherhood #14)       5    4861.44
      Fantasy                                                                                                              The Star-Touched 

## Wrap up

In [11]:
conn.close()
print("DONE. Database saved at:", os.path.abspath(DB_PATH))
print("Rows in final dataset:", len(clean_df), "| Categories:", clean_df['category'].nunique())

DONE. Database saved at: c:\Users\Srivatsav\Downloads\data-pipeline\books.db
Rows in final dataset: 159 | Categories: 4
